# Form Catalog

This notebook is the reference for what is stored on the IMQCAM DMS. It walks the form registry, pulls each form's entries with a targeted query, and profiles them field by field — name, fill rate, and observed types — so you can tell at a glance which columns are usable before writing any analysis.

It covers:

- The form inventory with entry counts
- Per-form field profiles, including how sparse each field is
- Which fields are nested objects and which are lists of repeated measurements
- How each form identifies its sample, and which ones can be joined

## Setup

`/form` gives the registry; each form's entries are then fetched with `formId` so nothing irrelevant crosses the wire.

In [1]:
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import entries_to_frame, explode_records, fetch_entries, get_client, list_forms

client = get_client()
forms = list_forms(client)

entries_by_form = {
    form_id: fetch_entries(client, form_id=form_id, page_size=100)
    for form_id in forms
}

print(f"{len(forms)} forms, {sum(len(e) for e in entries_by_form.values())} entries")

9 forms, 758 entries


## Form inventory

The spread is wide — three forms hold three quarters of the data, and the two richest schemas (Printer Build, Sample Heat Treatment Record) have the fewest entries.

In [2]:
inventory = pd.DataFrame(
    [
        {"form": forms[form_id], "formId": form_id, "entries": len(form_entries)}
        for form_id, form_entries in entries_by_form.items()
    ]
).sort_values("entries", ascending=False).reset_index(drop=True)

inventory

,form,formId,entries
0,NASA Ti64 30um Layers Data,67d39472366ec49ab59dd4db,201
1,Four-point flexural test,68922e94f5b193b7d3e07f5c,193
2,Archival ULI Build (simplified),68922e35f5b193b7d3e07f5b,192
3,Fractography Record,69fa1d0872a32de5fe95028b,72
4,Micromechanical Simulation Data,68ed0eb251d2cf4a1aa1d14f,46
5,AM Build Parameters,6970da157f6ebb8fb320705c,27
6,Raw Powder Details,663e6d21b18fa1c426e939ab,11
7,Sample Heat Treatment Record,69fa18be514cf501621588cc,9
8,Printer Build,66425a71b18fa1c426e93aa0,7


## Field profiles

For each form: every field under `data`, how many entries carry it, and what types were observed. Nested objects are shown with dotted paths; lists are reported as `list` with the range of lengths seen.

A field present on every entry is safe to rely on. Anything below 100% needs a fill or a filter before it reaches a model.

In [3]:
def describe_value(value):
    """Return a short type label for one value."""
    if isinstance(value, list):
        return "list"
    return type(value).__name__


def flatten(payload, prefix=""):
    """Flatten nested dicts to dotted paths, stopping at lists."""
    flat = {}
    for key, value in payload.items():
        path = prefix + key
        if isinstance(value, dict):
            flat.update(flatten(value, path + "."))
        else:
            flat[path] = value
    return flat


def profile(form_entries):
    """Field-level profile for one form's entries."""
    present = Counter()
    types = defaultdict(set)
    list_lengths = defaultdict(list)

    for entry in form_entries:
        payload = entry.get("data") or {}
        for path, value in flatten(payload).items():
            present[path] += 1
            types[path].add(describe_value(value))
            if isinstance(value, list):
                list_lengths[path].append(len(value))

    rows = []
    for path, n in present.most_common():
        length_note = ""
        if path in list_lengths:
            lengths = list_lengths[path]
            length_note = f"len {min(lengths)}-{max(lengths)}"
        rows.append(
            {
                "field": path,
                "present": n,
                "fill": f"{n / len(form_entries):.0%}",
                "types": ", ".join(sorted(types[path])),
                "notes": length_note,
            }
        )
    return pd.DataFrame(rows)

### Profiles, form by form

Printed rather than returned as a frame so all nine fit in one output.

In [4]:
pd.set_option("display.max_rows", 200)

for _, row in inventory.iterrows():
    form_entries = entries_by_form[row["formId"]]
    print("=" * 78)
    print(f"{row['form']}  [{row['formId']}]  n={len(form_entries)}")
    print("=" * 78)
    frame = profile(form_entries)
    if frame.empty:
        print("  (no fields)\n")
        continue
    for _, field in frame.iterrows():
        print(f"  {field['fill']:>4}  {field['field']:<52} "
              f"{field['types']:<18} {field['notes']}")
    print()

NASA Ti64 30um Layers Data  [67d39472366ec49ab59dd4db]  n=201
  100%  Build_Date                                           str                
  100%  Build_ID                                             str                
  100%  DOE_Code                                             int                
  100%  DOE_ID                                               str                
  100%  Elongation_Percent                                   float, int         
  100%  Geometry                                             str                
  100%  Hatch_mm                                             float              
  100%  Layer_mm                                             float              
  100%  Location                                             str                
  100%  Material                                             str                
  100%  Modulus_GPa                                          float, int         
  100%  Orientation                            

## Fields holding repeated measurements

Several forms store lists of dicts under `data`. Each element is a separate measurement, so one entry is many rows — this is the single most important thing to know before analysing them.

`explode_records()` in `imqcam.py` turns any of these into a tidy frame, one row per element, carrying the parent's `uniqueId` and IGSN down.

In [5]:
LIST_FIELDS = {
    "tests": "one fatigue test / one measured defect",
    "buildParameters": "one laser parameter set per scan type",
    "composition": "one alloying element",
    "buildGeometries": "one specimen geometry and its count",
    "heat_treatment": "one heat-treatment step",
}

all_entries = [e for form_entries in entries_by_form.values() for e in form_entries]

rows = []
for field, meaning in LIST_FIELDS.items():
    exploded = explode_records(all_entries, field)
    if exploded.empty:
        continue
    source_forms = sorted({forms[f] for f in exploded["formId"].unique() if f in forms})
    rows.append(
        {
            "field": field,
            "rows when exploded": len(exploded),
            "parent entries": exploded["uniqueId"].nunique(),
            "each row is": meaning,
            "forms": ", ".join(source_forms),
        }
    )

pd.DataFrame(rows)

,field,rows when exploded,parent entries,each row is,forms
0,tests,533,265,one fatigue test / one measured defect,"Four-point flexural test, Fractography Record"
1,buildParameters,33,27,one laser parameter set per scan type,AM Build Parameters
2,composition,132,11,one alloying element,Raw Powder Details
3,buildGeometries,18,7,one specimen geometry and its count,Printer Build
4,heat_treatment,12,7,one heat-treatment step,Sample Heat Treatment Record


## How samples are identified

Cross-form analysis needs a join key, and the DMS records the sample IGSN four different ways depending on the form:

- **`assignedIGSN`** — Printer Build
- **`sampleIGSN`** — Fractography Record
- **`lookup`** as a `"IGSN - depositionId - buildId"` string — ULI, four-point flexural
- **`lookup`** as a list of IGSNs — AM Build Parameters, Sample Heat Treatment Record

`extract_igsn()` handles all four.

In [6]:
frame = entries_to_frame(all_entries)
coverage = (
    frame.groupby("formName")
    .agg(entries=("_id", "count"), with_igsn=("igsn", lambda s: s.notna().sum()))
)
coverage["igsn coverage"] = (coverage["with_igsn"] / coverage["entries"]).map("{:.0%}".format)
coverage.sort_values("entries", ascending=False)

,entries,with_igsn,igsn coverage
formName,,,
NASA Ti64 30um Layers Data,201,0,0%
Four-point flexural test,193,192,99%
Archival ULI Build (simplified),192,192,100%
Fractography Record,72,72,100%
Micromechanical Simulation Data,46,0,0%
AM Build Parameters,27,27,100%
Raw Powder Details,11,0,0%
Sample Heat Treatment Record,9,9,100%
Printer Build,7,7,100%


Three groups come out at 0%, for different reasons:

- **NASA TTT** (201 entries) is the consequential one. It holds the richest mechanical data on the DMS — yield strength, elongation, modulus, porosity — and none of it can currently be tied to a physical sample tracked elsewhere. It identifies builds by `Build_ID` and `Parameter_Label` only.
- **Raw Powder Details** identifies feedstock by `rawPowderID`, not by sample IGSN. Printer Build points back at it through `rawPowderID`, so the link exists in that direction.
- **Micromechanical Simulation Data** describes simulations, not physical specimens, so there is no IGSN to carry.

Of the forms that do use IGSNs, only the four-point flexural form falls short of complete: 192 of 193. The remaining entry has a `lookup` that does not parse to an IGSN.